# QSOLVE — Single Quantum Route Demonstration

## One truck, two factories, one quantum routing decision

This notebook isolates **one multi-factory truck route** from the fertilizer-distribution problem.

### Route subproblem

A 10-tonne truck leaves **Karatina** and must deliver:

- **Ragati:** 8.380 tonnes
- **Nguguini:** 1.620 tonnes

Total:

\[
8.380 + 1.620 = 10.000\text{ tonnes}
\]

There are only two possible route orders:

\[
|0\rangle:
Karatina \rightarrow Ragati \rightarrow Nguguini \rightarrow Karatina
\]

\[
|1\rangle:
Karatina \rightarrow Nguguini \rightarrow Ragati \rightarrow Karatina
\]

The quantum task is to identify the lower-cost route.

This is intentionally a **small 1-qubit demonstration**, following the mentor recommendation to begin with the smallest instance before scaling.

## Objective

The transport objective is load-dependent:

\[
C = 16\sum_{(i,j)} d_{ij}L_{ij}
\]

where:

- \(d_{ij}\) = road distance in km;
- \(L_{ij}\) = fertilizer carried on that road leg in tonnes;
- 16 = KSh per tonne-km.

After all fertilizer has been delivered, the return leg carries 0 tonnes and therefore contributes zero freight tonne-km cost under the current cost interpretation.

In [ ]:
# Run once in qBraid Lab.
# Restart the kernel if requested after installation.

%pip install -q "qiskit>=2.3,<3" "qbraid>=0.12" numpy pandas scipy matplotlib openpyxl

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import DiagonalGate

print("Environment ready.")

## 1. Load the single-route dataset

In [ ]:
DATA_FILE = "QSOLVE_Single_Quantum_Route_Dataset.xlsx"

DEPOT = "Karatina"
FACTORIES = ["Ragati", "Nguguini"]

# Embedded fallback values
DELIVERIES_FALLBACK = {
    "Ragati": 8.380,
    "Nguguini": 1.620,
}

DIST_FALLBACK = pd.DataFrame(
    [
        [0.00, 11.82, 18.92],
        [11.82, 0.00, 15.86],
        [18.92, 15.86, 0.00],
    ],
    index=["Karatina", "Ragati", "Nguguini"],
    columns=["Karatina", "Ragati", "Nguguini"],
)

TIME_FALLBACK = pd.DataFrame(
    [
        [0.00, 0.27, 0.44],
        [0.27, 0.00, 0.38],
        [0.44, 0.38, 0.00],
    ],
    index=["Karatina", "Ragati", "Nguguini"],
    columns=["Karatina", "Ragati", "Nguguini"],
)

TRUCK_CAPACITY = 10.0
TRANSPORT_RATE = 16.0
MAX_ROUTE_TIME = 8.0


if os.path.exists(DATA_FILE):
    print("Loading:", DATA_FILE)

    inputs_df = pd.read_excel(
        DATA_FILE,
        sheet_name="Route_Inputs"
    )

    DIST = pd.read_excel(
        DATA_FILE,
        sheet_name="Distance_Matrix",
        index_col=0
    ).astype(float)

    TIME = pd.read_excel(
        DATA_FILE,
        sheet_name="Time_Matrix",
        index_col=0
    ).astype(float)

    factory_rows = inputs_df[
        inputs_df["Type"].str.lower() == "factory"
    ]

    DELIVERIES = dict(
        zip(
            factory_rows["Location"],
            factory_rows["Delivery_t"].astype(float)
        )
    )

else:
    print("Dataset file not found. Using embedded values.")

    DELIVERIES = DELIVERIES_FALLBACK.copy()
    DIST = DIST_FALLBACK.copy()
    TIME = TIME_FALLBACK.copy()


TOTAL_LOAD = sum(DELIVERIES.values())

assert abs(TOTAL_LOAD - 10.0) < 1e-9

print("\nDeliveries:")
for factory, qty in DELIVERIES.items():
    print(f"{factory:<12}: {qty:.3f} t")

print("\nTotal truck load:", TOTAL_LOAD, "t")

display(DIST)
display(TIME)

## 2. Calculate the two possible routes exactly

In [ ]:
def evaluate_route(order):
    current = DEPOT
    load = TOTAL_LOAD

    total_distance = 0.0
    total_time = 0.0
    total_tonne_km = 0.0
    total_cost = 0.0

    legs = []

    for factory in order:
        d = float(DIST.loc[current, factory])
        t = float(TIME.loc[current, factory])

        tonne_km = load * d
        leg_cost = TRANSPORT_RATE * tonne_km

        delivered = DELIVERIES[factory]

        legs.append({
            "From": current,
            "To": factory,
            "Load_t": load,
            "Delivered_t": delivered,
            "Distance_km": d,
            "Time_h": t,
            "Tonne_km": tonne_km,
            "Cost_KSh": leg_cost,
        })

        total_distance += d
        total_time += t
        total_tonne_km += tonne_km
        total_cost += leg_cost

        load -= delivered
        current = factory

    # Empty return
    d = float(DIST.loc[current, DEPOT])
    t = float(TIME.loc[current, DEPOT])

    tonne_km = load * d
    leg_cost = TRANSPORT_RATE * tonne_km

    legs.append({
        "From": current,
        "To": DEPOT,
        "Load_t": load,
        "Delivered_t": 0.0,
        "Distance_km": d,
        "Time_h": t,
        "Tonne_km": tonne_km,
        "Cost_KSh": leg_cost,
    })

    total_distance += d
    total_time += t
    total_tonne_km += tonne_km
    total_cost += leg_cost

    return {
        "order": list(order),
        "distance_km": total_distance,
        "time_h": total_time,
        "tonne_km": total_tonne_km,
        "cost_ksh": total_cost,
        "remaining_load_t": load,
        "legs": legs,
    }


ROUTE_0 = ["Ragati", "Nguguini"]
ROUTE_1 = ["Nguguini", "Ragati"]

metrics_0 = evaluate_route(ROUTE_0)
metrics_1 = evaluate_route(ROUTE_1)

candidate_routes_df = pd.DataFrame([
    {
        "Quantum_State": "|0>",
        "Route": "Karatina -> Ragati -> Nguguini -> Karatina",
        "Distance_km": metrics_0["distance_km"],
        "Time_h": metrics_0["time_h"],
        "Tonne_km": metrics_0["tonne_km"],
        "Cost_KSh": metrics_0["cost_ksh"],
    },
    {
        "Quantum_State": "|1>",
        "Route": "Karatina -> Nguguini -> Ragati -> Karatina",
        "Distance_km": metrics_1["distance_km"],
        "Time_h": metrics_1["time_h"],
        "Tonne_km": metrics_1["tonne_km"],
        "Cost_KSh": metrics_1["cost_ksh"],
    },
])

display(candidate_routes_df)

In [ ]:
best_classical_row = candidate_routes_df.loc[
    candidate_routes_df["Cost_KSh"].idxmin()
]

print("Exact best route:")
print(best_classical_row["Route"])

print(
    "Exact minimum transport cost:",
    f"KSh {best_classical_row['Cost_KSh']:,.2f}"
)

## 3. Quantum encoding

We use a single qubit:

\[
|0\rangle =
Karatina \rightarrow Ragati \rightarrow Nguguini \rightarrow Karatina
\]

\[
|1\rangle =
Karatina \rightarrow Nguguini \rightarrow Ragati \rightarrow Karatina
\]

The two route costs become the two diagonal energies of the cost Hamiltonian.

For stable QAOA optimization, the raw costs are normalized to the interval \([0,1]\). This does **not** change which route is optimal.

In [ ]:
RAW_COSTS = np.array([
    metrics_0["cost_ksh"],
    metrics_1["cost_ksh"],
], dtype=float)

COST_MIN = RAW_COSTS.min()
COST_MAX = RAW_COSTS.max()

NORMALIZED_COSTS = (
    RAW_COSTS - COST_MIN
) / (
    COST_MAX - COST_MIN
)

print("Raw route costs:", RAW_COSTS)
print("Normalized Hamiltonian energies:", NORMALIZED_COSTS)

# For these two routes this becomes approximately diag(0, 1).

For a one-qubit diagonal Hamiltonian:

\[
H_C =
\begin{bmatrix}
E_0 & 0\\
0 & E_1
\end{bmatrix}
\]

and because \(E_0=0\), \(E_1=1\) after normalization:

\[
H_C = \frac{I-Z}{2}.
\]

QAOA starts from:

\[
|+\rangle
=
\frac{|0\rangle+|1\rangle}{\sqrt{2}}
\]

and alternates a cost phase and an \(X\)-mixer.

## 4. Build the one-qubit QAOA circuit

In [ ]:
def build_qaoa_circuit(gamma, beta, measure=False):
    circuit = QuantumCircuit(1)

    # Initial |+> state
    circuit.h(0)

    # Cost separator:
    # exp(-i gamma H_C)
    phases = np.exp(
        -1j * gamma * NORMALIZED_COSTS
    )

    circuit.append(
        DiagonalGate(phases.tolist()),
        [0]
    )

    # Mixer:
    # exp(-i beta X) = Rx(2 beta)
    circuit.rx(
        2.0 * beta,
        0
    )

    if measure:
        circuit.measure_all()

    return circuit


test_circuit = build_qaoa_circuit(
    gamma=1.0,
    beta=0.5,
    measure=False
)

print(test_circuit)
print("Qubits:", test_circuit.num_qubits)
print("Circuit depth:", test_circuit.depth())

## 5. QAOA objective function

In [ ]:
def qaoa_probabilities(params):
    gamma, beta = params

    circuit = build_qaoa_circuit(
        gamma,
        beta,
        measure=False
    )

    state = Statevector.from_instruction(
        circuit
    )

    probabilities = np.abs(
        state.data
    ) ** 2

    return probabilities


def qaoa_expected_energy(params):
    probabilities = qaoa_probabilities(
        params
    )

    return float(
        np.dot(
            probabilities,
            NORMALIZED_COSTS
        )
    )

## 6. Optimize the QAOA parameters

In [ ]:
RESTART_SEEDS = [7, 19, 42]
MAXITER = 150

best_result = None
best_seed = None

start_time = time.perf_counter()

for seed in RESTART_SEEDS:
    rng = np.random.default_rng(seed)

    initial_point = np.array([
        rng.uniform(0.0, 2.0 * np.pi),  # gamma
        rng.uniform(0.0, np.pi),        # beta
    ])

    result = minimize(
        qaoa_expected_energy,
        initial_point,
        method="COBYLA",
        options={
            "maxiter": MAXITER,
            "rhobeg": 0.5,
        }
    )

    print(
        f"Seed {seed}: "
        f"energy={result.fun:.8f}, "
        f"gamma={result.x[0]:.6f}, "
        f"beta={result.x[1]:.6f}"
    )

    if best_result is None or result.fun < best_result.fun:
        best_result = result
        best_seed = seed

runtime = time.perf_counter() - start_time

OPT_GAMMA = float(best_result.x[0])
OPT_BETA = float(best_result.x[1])

print("\nBest seed:", best_seed)
print("Optimal gamma:", OPT_GAMMA)
print("Optimal beta:", OPT_BETA)
print("Minimum expected normalized energy:", best_result.fun)
print("Optimization runtime:", runtime, "seconds")

## 7. Decode the quantum result

In [ ]:
final_probabilities = qaoa_probabilities(
    [OPT_GAMMA, OPT_BETA]
)

P_ROUTE_0 = float(final_probabilities[0])
P_ROUTE_1 = float(final_probabilities[1])

print("Probability |0>:", P_ROUTE_0)
print("Probability |1>:", P_ROUTE_1)

selected_state = int(
    np.argmax(final_probabilities)
)

if selected_state == 0:
    selected_route = candidate_routes_df.iloc[0]
else:
    selected_route = candidate_routes_df.iloc[1]

print("\nMost probable quantum route:")
print(selected_route["Route"])

print(
    "Transport cost:",
    f"KSh {selected_route['Cost_KSh']:,.2f}"
)

print(
    "Probability:",
    f"{100 * max(final_probabilities):.2f}%"
)

## 8. Emulate quantum measurements

In [ ]:
SHOTS = 4096
SHOT_SEED = 2026

rng = np.random.default_rng(
    SHOT_SEED
)

samples = rng.choice(
    [0, 1],
    size=SHOTS,
    p=final_probabilities
)

count_0 = int(
    np.sum(samples == 0)
)

count_1 = int(
    np.sum(samples == 1)
)

counts_df = pd.DataFrame({
    "Quantum_State": ["|0>", "|1>"],
    "Counts": [count_0, count_1],
    "Measured_Probability": [
        count_0 / SHOTS,
        count_1 / SHOTS
    ],
    "Route": [
        candidate_routes_df.iloc[0]["Route"],
        candidate_routes_df.iloc[1]["Route"],
    ],
    "Cost_KSh": [
        candidate_routes_df.iloc[0]["Cost_KSh"],
        candidate_routes_df.iloc[1]["Cost_KSh"],
    ],
})

display(counts_df)

In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    ["|0>\nRagati first", "|1>\nNguguini first"],
    [count_0, count_1]
)

plt.ylabel("Measurement counts")
plt.title("QAOA measurement results — single route")
plt.show()

## 9. Compare quantum result with the exact solution

In [ ]:
exact_cost = float(
    best_classical_row["Cost_KSh"]
)

quantum_cost = float(
    selected_route["Cost_KSh"]
)

approximation_ratio = (
    quantum_cost / exact_cost
)

matched_optimum = (
    abs(quantum_cost - exact_cost) < 1e-9
)

comparison_df = pd.DataFrame([{
    "Factories": 2,
    "Qubits": 1,
    "Possible_Routes": 2,
    "Exact_Route": best_classical_row["Route"],
    "Exact_Cost_KSh": exact_cost,
    "QAOA_Route": selected_route["Route"],
    "QAOA_Cost_KSh": quantum_cost,
    "Approximation_Ratio": approximation_ratio,
    "QAOA_Matched_Exact_Optimum": matched_optimum,
    "Probability_of_Selected_Route": max(final_probabilities),
    "Shots": SHOTS,
    "Runtime_seconds": runtime,
}])

display(comparison_df)

## 10. Leg-by-leg details of the selected route

In [ ]:
selected_order = (
    ROUTE_0
    if selected_state == 0
    else ROUTE_1
)

selected_metrics = evaluate_route(
    selected_order
)

leg_df = pd.DataFrame(
    selected_metrics["legs"]
)

display(leg_df)

print(
    "Total route distance:",
    f"{selected_metrics['distance_km']:.2f} km"
)

print(
    "Total route time:",
    f"{selected_metrics['time_h']:.2f} h"
)

print(
    "Total transport cost:",
    f"KSh {selected_metrics['cost_ksh']:,.2f}"
)

print(
    "Remaining fertilizer on return:",
    f"{selected_metrics['remaining_load_t']:.3f} t"
)

## 11. Constraint validation

In [ ]:
checks = {
    "Truck capacity satisfied":
        TOTAL_LOAD <= TRUCK_CAPACITY + 1e-9,

    "Route time <= 8 hours":
        selected_metrics["time_h"]
        <= MAX_ROUTE_TIME + 1e-9,

    "All fertilizer delivered":
        abs(
            selected_metrics["remaining_load_t"]
        ) < 1e-9,
}

for name, passed in checks.items():
    print(
        "✓" if passed else "✗",
        name
    )

assert all(checks.values())

## 12. Build the final measured Qiskit circuit

In [ ]:
final_circuit = build_qaoa_circuit(
    OPT_GAMMA,
    OPT_BETA,
    measure=True
)

print(final_circuit)

print(
    "Logical qubits:",
    final_circuit.num_qubits
)

print(
    "Logical circuit depth:",
    final_circuit.depth()
)

## 13. Optional qBraid cloud execution

The local cells above are enough to complete the experiment.

If you also want to submit the optimized one-qubit circuit to the qBraid QIR simulator, uncomment the following cell.

In [ ]:
# OPTIONAL qBRAID EXECUTION
#
# from qbraid import QbraidProvider
#
# provider = QbraidProvider()
#
# device = provider.get_device(
#     "qbraid:qbraid:sim:qir-sv"
# )
#
# job = device.run(
#     final_circuit,
#     shots=SHOTS
# )
#
# print("Job:", job)
#
# cloud_result = job.result()
#
# cloud_counts = (
#     cloud_result
#     .data
#     .get_counts()
# )
#
# print("qBraid counts:")
# print(cloud_counts)

## 14. Export the single-route results

In [ ]:
OUTPUT_FILE = "QSOLVE_Single_Route_QAOA_Results.xlsx"

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    candidate_routes_df.to_excel(
        writer,
        sheet_name="Candidate_Routes",
        index=False
    )

    counts_df.to_excel(
        writer,
        sheet_name="Quantum_Counts",
        index=False
    )

    comparison_df.to_excel(
        writer,
        sheet_name="Quantum_vs_Exact",
        index=False
    )

    leg_df.to_excel(
        writer,
        sheet_name="Selected_Route_Legs",
        index=False
    )

print("Saved:", OUTPUT_FILE)

# Interpretation for the presentation

This experiment should be described as:

> “We began with the smallest multi-factory instance: one truck, two factories and two possible route orders. We encoded those two routes in one qubit and used QAOA to bias the quantum state toward the lower load-dependent transport-cost route. The result was then checked against the exact classical enumeration.”

### Important limitation

With only two factories, classical enumeration is trivial. This experiment is a **proof of formulation and reproducibility**, not evidence of quantum advantage.

The next scientific step is to scale the exact same logic to 3, 4, 5, 6 and 7 factories and observe how qubits, route space, runtime and solution quality change.